# Baseline Models For Price Prediction

This notebook builds first benchmark models for the current same-time explanatory task:

`boundary_actual_values(t) -> price(t)`

It trains Linear Regression and Ridge Regression using the reduced feature set that keeps `Wind_And_Solar` and drops `Wind_Power` and `Photovoltaic`.

In [ ]:
from pathlib import Path
import os


def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "output/clean_datasets/price_prediction_dataset.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find output/clean_datasets/price_prediction_dataset.csv")


PROJECT_ROOT = find_project_root()
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".matplotlib-cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

INPUT_PATH = PROJECT_ROOT / "output/clean_datasets/price_prediction_dataset.csv"
OUTPUT_DIR = PROJECT_ROOT / "output/baseline_results/price_prediction"

FEATURE_COLUMNS = [
    "System_Load",
    "Wind_And_Solar",
    "Tie_Line",
    "Hydropower",
    "Non-Marketized_Unit",
]
TARGET_COLUMN = "price"

TRAIN_END = pd.Timestamp("2025-10-31 23:45:00")
VAL_END = pd.Timestamp("2025-11-30 23:45:00")

plt.rcParams["figure.dpi"] = 130
plt.rcParams["savefig.dpi"] = 200
plt.rcParams["font.size"] = 10

print(f"Project root: {PROJECT_ROOT}")
print(f"Input dataset: {INPUT_PATH}")
print(f"Output directory: {OUTPUT_DIR}")

## Load Data

In [ ]:
df = pd.read_csv(INPUT_PATH, parse_dates=["time"])
df = df.sort_values("time").reset_index(drop=True)

required_columns = ["time", *FEATURE_COLUMNS, TARGET_COLUMN]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

model_df = df[required_columns].copy()

print(f"Rows: {len(model_df):,}")
print(f"Time range: {model_df['time'].min()} to {model_df['time'].max()}")
print(f"Features: {FEATURE_COLUMNS}")
display(model_df.head())
display(model_df[FEATURE_COLUMNS + [TARGET_COLUMN]].describe().T)

## Chronological Split

Because this is time-ordered data, split chronologically instead of randomly.

- Train: January through October 2025
- Validation: November 2025
- Test: December 2025

In [ ]:
train_df = model_df[model_df["time"] <= TRAIN_END].copy()
val_df = model_df[(model_df["time"] > TRAIN_END) & (model_df["time"] <= VAL_END)].copy()
test_df = model_df[model_df["time"] > VAL_END].copy()

split_summary = pd.DataFrame(
    {
        "train": {"rows": len(train_df), "start": train_df["time"].min(), "end": train_df["time"].max()},
        "validation": {"rows": len(val_df), "start": val_df["time"].min(), "end": val_df["time"].max()},
        "test": {"rows": len(test_df), "start": test_df["time"].min(), "end": test_df["time"].max()},
    }
).T
display(split_summary)

X_train = train_df[FEATURE_COLUMNS]
y_train = train_df[TARGET_COLUMN]
X_val = val_df[FEATURE_COLUMNS]
y_val = val_df[TARGET_COLUMN]
X_test = test_df[FEATURE_COLUMNS]
y_test = test_df[TARGET_COLUMN]

X_train_val = pd.concat([X_train, X_val], axis=0)
y_train_val = pd.concat([y_train, y_val], axis=0)

## Helpers

In [ ]:
def make_linear_model():
    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", LinearRegression()),
        ]
    )


def make_ridge_model(alpha):
    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=alpha)),
        ]
    )


def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }


def evaluate_model(model, model_name, datasets):
    rows = []
    for split_name, (X, y) in datasets.items():
        pred = model.predict(X)
        rows.append({"model": model_name, "split": split_name, **regression_metrics(y, pred)})
    return pd.DataFrame(rows)


def coefficient_table(model, model_name):
    estimator = model.named_steps["model"]
    return pd.DataFrame(
        {
            "model": model_name,
            "feature": FEATURE_COLUMNS,
            "coefficient_per_1_std": estimator.coef_,
        }
    ).sort_values("coefficient_per_1_std", key=lambda values: values.abs(), ascending=False)

## Linear Regression Baseline

In [ ]:
linear_val_model = make_linear_model()
linear_val_model.fit(X_train, y_train)

validation_datasets = {
    "train": (X_train, y_train),
    "validation": (X_val, y_val),
}

linear_validation_metrics = evaluate_model(
    linear_val_model,
    "Linear Regression",
    validation_datasets,
)

display(linear_validation_metrics)
display(coefficient_table(linear_val_model, "Linear Regression"))

## Ridge Regression Baseline

Ridge adds L2 regularization. The features are standardized inside the pipeline, then `alpha` is selected by validation RMSE.

In [ ]:
ridge_alphas = np.logspace(-4, 4, 17)
ridge_rows = []
ridge_models = {}

for alpha in ridge_alphas:
    model = make_ridge_model(alpha)
    model.fit(X_train, y_train)
    ridge_models[alpha] = model
    val_pred = model.predict(X_val)
    ridge_rows.append({"alpha": alpha, **regression_metrics(y_val, val_pred)})

ridge_search = pd.DataFrame(ridge_rows).sort_values("RMSE")
best_alpha = float(ridge_search.iloc[0]["alpha"])
ridge_val_model = ridge_models[best_alpha]

print(f"Best alpha by validation RMSE: {best_alpha:g}")
display(ridge_search)

ridge_validation_metrics = evaluate_model(
    ridge_val_model,
    f"Ridge Regression alpha={best_alpha:g}",
    validation_datasets,
)

display(ridge_validation_metrics)
display(coefficient_table(ridge_val_model, f"Ridge Regression alpha={best_alpha:g}"))

## Validation Comparison

In [ ]:
validation_comparison = pd.concat(
    [linear_validation_metrics, ridge_validation_metrics],
    ignore_index=True,
)

display(validation_comparison.sort_values(["split", "RMSE"]))

fig_alpha, ax = plt.subplots(figsize=(7, 4.5))
ridge_plot = ridge_search.sort_values("alpha")
ax.semilogx(ridge_plot["alpha"], ridge_plot["RMSE"], marker="o")
ax.axvline(best_alpha, color="#b2182b", linestyle="--", label=f"best alpha={best_alpha:g}")
ax.set_xlabel("Ridge alpha")
ax.set_ylabel("Validation RMSE")
ax.set_title("Ridge Alpha Search")
ax.legend()
fig_alpha.tight_layout()
display(fig_alpha)

## Final Test Evaluation

After selecting Ridge `alpha` on validation data, refit both models on train + validation and evaluate once on the December test set.

In [ ]:
linear_final_model = make_linear_model()
linear_final_model.fit(X_train_val, y_train_val)

ridge_final_model = make_ridge_model(best_alpha)
ridge_final_model.fit(X_train_val, y_train_val)

test_datasets = {"test": (X_test, y_test)}
test_metrics = pd.concat(
    [
        evaluate_model(linear_final_model, "Linear Regression", test_datasets),
        evaluate_model(ridge_final_model, f"Ridge Regression alpha={best_alpha:g}", test_datasets),
    ],
    ignore_index=True,
)

display(test_metrics.sort_values("RMSE"))

final_coefficients = pd.concat(
    [
        coefficient_table(linear_final_model, "Linear Regression"),
        coefficient_table(ridge_final_model, f"Ridge Regression alpha={best_alpha:g}"),
    ],
    ignore_index=True,
)
display(final_coefficients)

## Test Set Visual Checks

In [ ]:
test_predictions = test_df[["time", TARGET_COLUMN]].copy()
test_predictions["linear_pred"] = linear_final_model.predict(X_test)
test_predictions["ridge_pred"] = ridge_final_model.predict(X_test)
test_predictions["linear_residual"] = test_predictions[TARGET_COLUMN] - test_predictions["linear_pred"]
test_predictions["ridge_residual"] = test_predictions[TARGET_COLUMN] - test_predictions["ridge_pred"]

plot_window = test_predictions.head(7 * 24 * 4)

fig_pred, ax = plt.subplots(figsize=(11, 4.8))
ax.plot(plot_window["time"], plot_window[TARGET_COLUMN], label="Actual", color="#111111", linewidth=1.8)
ax.plot(plot_window["time"], plot_window["linear_pred"], label="Linear", color="#2166ac", alpha=0.85)
ax.plot(plot_window["time"], plot_window["ridge_pred"], label="Ridge", color="#b2182b", alpha=0.85)
ax.set_title("Test Predictions: First 7 Days Of December")
ax.set_xlabel("Time")
ax.set_ylabel("Standardized price")
ax.legend()
fig_pred.autofmt_xdate()
fig_pred.tight_layout()
display(fig_pred)

fig_resid, ax = plt.subplots(figsize=(7.5, 4.5))
ax.hist(test_predictions["linear_residual"], bins=40, alpha=0.65, label="Linear", color="#2166ac")
ax.hist(test_predictions["ridge_residual"], bins=40, alpha=0.65, label="Ridge", color="#b2182b")
ax.axvline(0, color="#111111", linewidth=1)
ax.set_title("Test Residual Distribution")
ax.set_xlabel("Actual - predicted")
ax.set_ylabel("Count")
ax.legend()
fig_resid.tight_layout()
display(fig_resid)

## Save Outputs

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

validation_comparison.to_csv(OUTPUT_DIR / "baseline_validation_metrics.csv", index=False)
test_metrics.to_csv(OUTPUT_DIR / "baseline_test_metrics.csv", index=False)
ridge_search.to_csv(OUTPUT_DIR / "ridge_alpha_search.csv", index=False)
final_coefficients.to_csv(OUTPUT_DIR / "baseline_coefficients.csv", index=False)
test_predictions.to_csv(OUTPUT_DIR / "baseline_test_predictions.csv", index=False)

fig_alpha.savefig(OUTPUT_DIR / "ridge_alpha_search.png", bbox_inches="tight")
fig_pred.savefig(OUTPUT_DIR / "baseline_test_predictions_first_week.png", bbox_inches="tight")
fig_resid.savefig(OUTPUT_DIR / "baseline_test_residuals.png", bbox_inches="tight")

print(f"Saved outputs to: {OUTPUT_DIR}")